# 팀 프로젝트2 — 14조: 다음 학년도 중도탈락 고위험 학과는 어디인가
학업지속 지원 대상 학과 선정을 위한 대학알리미 학과별 중도탈락 패널(2023~2026) 분석 및 예측

- 팀: 정승엽, 박유진, 신효주 · 담당교수 박주원
- Colab 정본 링크: `[팀 입력]`
- 실행 방법: CSV를 올린 뒤 **런타임 → 다시 시작 및 모두 실행** (처음부터 재실행 가능해야 함, 불가 시 −5점)
- 규칙: 모든 수치는 셀 출력으로 남기고 보고서·발표에는 셀 번호와 함께 옮긴다. 분석 코드는 `hypotheses-locked` 태그 이후에만 작성.

## 0. 환경·경로

In [ ]:
# 프롬프트 — (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] 실행 환경(Colab/로컬)
# [작업] 라이브러리 import, 경로 후보 탐색, 상수·스위치 정의
# [기준] Colab(/content/)과 로컬(../data/raw/) 후보 경로를 차례로 찾고 없으면 멈춤. RANDOM_STATE=42
# [보고] 찾은 경로, 라이브러리 버전

# TODO: import (pandas, numpy, matplotlib, koreanize_matplotlib, sklearn, statsmodels, scipy)
# TODO: 라이브러리 버전 출력 (재현성_확인표에 기록)
# TODO: CSV 경로 후보 리스트 → 존재하는 첫 경로 사용, 없으면 FileNotFoundError
# TODO: RANDOM_STATE = 42
# TODO: 되돌림 스위치 변수 (팀 결정 후 값 확정, 정제로그 6절과 이름 일치)
#   MIN_ENROLL   : 직전연도 최소 재적 (사전 고정 30, 민감도 10·50)
#   RATE_SOURCE  : 공시 비율 사용 vs 계÷재적 재계산 (재계산 불일치 68행 처리)
#   THRESHOLD    : 0.5 vs 지원 가능 학과 수(k) 기준
# TODO: FIG_DIR = 'figures/' 생성, 그림 저장 함수

## ① 문제 정의 (15점)
> 아래는 `00_문제정의_가설.md`(git 태그 `hypotheses-locked`, 2026-11-08 고정) 인용. 이 셀은 수정하지 않는다.

- **누가**: 대학 학사지원 담당자 (적용 사례: 한국외대 학사종합지원센터, 입학처)
- **무엇을**: 다음 학년도 학업지속 지원(상담·학습지원·복학 안내)을 우선 배치할 학과
- **질문**: 직전 공시연도 정보로 다음 공시연도 중도탈락 고위험(상위 25%) 학과를 맞힐 수 있는가, '전년도 고위험 학과 그대로 지원' 기준선보다 얼마나 나은가 (분류)
- **H1 구조**: 비수도권·사립·소규모(재적 하위 25%) 학과는 고위험일 가능성이 높다 → 오즈비 > 1이고 95% CI가 1을 포함하지 않으면 맞음
- **H2 사유**: 탈락 인원의 과반은 자퇴·미등록·미복학 → 세 사유 합 ÷ 계 ≥ 50%(전체·계열별)이면 맞음
- **H3 예측**: 전년도 비율이 가장 강한 신호, 구조 정보 추가 개선은 작다 → 같은 k에서 적중률 개선 5%p 미만이면 맞음
- **H4 한국외대**: 소규모 어문학과군('기타유럽어・문학', '기타아시아어・문학')의 2024~2026 합산 탈락률이 ⓐ 한국외대 나머지 ⓑ 전국 동일 소계열보다 높다 → 둘 다 높고 부트스트랩 95% 구간 하한 > 0이면 맞음, 한쪽만이면 부분, 둘 다 아니면 기각
- **반대 결과**: 그대로 받아들이고 결론에 맞았다/틀렸다를 명시

## ② 수집·데이터 이해

In [ ]:
# 프롬프트 ② (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] data/raw/ 중도탈락 CSV (utf-8-sig)
# [작업] 원본 읽기, 출처·수집일 표기, 행·열·연도별 행 수, 컬럼 dtype과 측정 척도 표
# [기준] 원본 df는 수정하지 않음. 계획서 기준값(56,340×32, 연도별 14,350/13,628/13,928/14,434)과 assert 대조
# [보고] 수집 기록표·척도 표 (docs/수집_기록표.md와 일치 확인)

# TODO: 출처 표기 (공공데이터포털 15164570, 교육부 교육데이터정책과, 공공누리 제1유형, 포털 수정일 2026-09-10, 다운로드일 [팀 입력])
# TODO: df_raw = 원본 읽기 (encoding='utf-8-sig')
# TODO: assert df_raw.shape == (56340, 32)
# TODO: 공시연도별 행 수 → assert 기대값
# TODO: 컬럼별 dtype, 고유값 수, 예시값 표
# TODO: 측정 척도 표 (명목/구간/비율)

## ③ 정제 (`logs/정제로그.md` 순서 0~8)

In [ ]:
# 프롬프트 ③ (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] 원본 df 전체 / 연결 쌍 / 분석 표본
# [작업] 이슈별 표시(flag) 컬럼을 만들고 단계별 추적표를 같은 기준으로 센다
# [기준] 행을 지우지 않고 표시 후 표본에서 제외. y(비율) 보정 금지. 건수는 코드 계산값만. 기대값은 assert, 0행 필터는 멈춤
# [보고] 처리·건수·근거·영향 표, 원본 → 중복 제거 후 → 연결 쌍 → 분석 표본 추적표

# TODO: 단계별 추적 도구
#   make_flags(d)  : 이슈별 불리언 표 (재적=0, 재적=0&계>0, 계>재적, 재계산 불일치, 사유합≠계, 비율 결측, 계 결측, 폐교, 신설)
#   count_table(d) : 이슈 × 단계(원본 / 연결 쌍 / 분석 표본) 건수
#   log = []       : 처리·건수·근거·영향 행을 코드로 쌓음

In [ ]:
# ③-0 사전 진단(읽기 전용)
# 내용: 행·열 수, 연도별 행 수, 이슈별 건수
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 56,340×32, 연도별 14,350/13,628/13,928/14,434
# TODO

In [ ]:
# ③-1 공백·표기
# 내용: 컬럼명·문자열 값 공백 점검 (학과명 앞뒤 공백 1건 확인됨)
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): strip 후 컬럼 32개
# TODO

In [ ]:
# ③-2 타입 확인
# 내용: 숫자 컬럼이 숫자형인지, 변환 시 새 결측이 생기지 않는지
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 변환 전후 NaN 증가 0
# TODO

In [ ]:
# ③-3 중복
# 내용: 완전 중복과 키(공시연도·학교코드·학과코드) 중복
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 0 / 0 — 0이면 '처리 없음'으로 로그
# TODO

In [ ]:
# ③-4 논리 모순 표시(삭제 금지)
# 내용: 재적=0, 재적=0이면서 계>0, 계>재적, 공시 비율 vs 계÷재적 불일치, 사유합≠계
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 2,500 / 1,497 / 172 / 68 / 0
# TODO

In [ ]:
# ③-5 결측 분해
# 내용: 비율 결측을 '계 결측'과 '재적 0'으로 분해, 연도별 결측률 비교. 보정하지 않음
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 6,372 = 5,569 + 803
# TODO

In [ ]:
# ③-6 패널 연결
# 내용: (학교코드, 학과코드)로 전년→당해 쌍 생성. 폐교·신설 제외. 한국외대 코드 변동 기록
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 쌍 29,071 (재적>0·비율 유효 조건)
# TODO

In [ ]:
# ③-7 분석 표본 확정
# 내용: 직전연도 재적≥MIN_ENROLL 등 포함 범위 적용, 행 수 단계별 표시. 10·50명 민감도 표본도 생성
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 단계별 추적표
# TODO

In [ ]:
# ③-8 손대지 않는 것·최종 점검
# 내용: 100% 초과 값 수정 안 함(표시만), 사유 보정 안 함 등 이유 기록. 필터 전후 행 수, 0행 무음 실패 확인, 정제 로그 출력
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): assert 통과
# TODO

In [ ]:
# ③-정제 결과 저장
# TODO: 분석 표본을 data/processed/ 에 저장 (코드로만 생성, 수동 편집 금지)
# TODO: 정제 로그 표(log)를 출력 → logs/정제로그.md 4절에 옮김

## ④ 분석·예측 (25점 — 기준선 비교 필수)

In [ ]:
# 프롬프트 ④ (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] 분석 표본 (학습 t=2024·2025 / 테스트 t=2026)
# [작업] EDA·기술통계 → 로지스틱 회귀 → 기준선 비교 → 민감도 → H1~H4 판정
# [기준] 무작위 분할 금지, RANDOM_STATE=42. 당해 계·사유·비율 사용 금지(누수). 75백분위 컷은 학습 데이터로만. 같은 k 정밀도 비교. 임계값 근거·오차 비용 명시
# [보고] 지표(정확도·정밀도·재현율·AUC·혼동행렬)를 기준선과 한 표로, 오즈비와 95% CI, 같은 k 적중률 차이(%p)

In [ ]:
# ④-1 EDA·기술통계 (재적·계 분포, 절대=인원과 비중=비율을 함께)
# TODO

In [ ]:
# ④-2 y 정의: 학습기간 유효 학과 75백분위 컷 계산 → 테스트에 그대로 적용
# TODO

In [ ]:
# ④-3 설명 변수: 전년도 비율, 전년도 재적(로그), 전년도 사유 구성비, 설립유형, 소재지유형, 표준분류 대계열, 주야간구분, 학교종류 / 누수 점검 assert
# TODO

In [ ]:
# ④-4 로지스틱 회귀 (statsmodels로 오즈비·95% CI, sklearn으로 예측 확률)
# TODO

In [ ]:
# ④-5 기준선 ① 전원 '고위험 아님' ② 전년도 고위험 지속 → 같은 k에서 정밀도 비교
# TODO

In [ ]:
# ④-6 민감도: 최소 재적 10·30·50 / 임계값 0.5 vs 지원 가능 학과 수(비용 가정 명시)
# TODO

In [ ]:
# ④-7 H1 판정 (비수도권·사립·소규모 오즈비, 95% CI)
# TODO

In [ ]:
# ④-8 H2 판정 (자퇴+미등록+미복학 ÷ 계, 전체·계열별)
# TODO

In [ ]:
# ④-9 H3 판정 (같은 k 적중률 개선 < 5%p ?, 민감도 표에서 결론 뒤집힘 여부)
# TODO

In [ ]:
# ④-10 H4 판정 (한국외대 어문군 2024~2026 합산 비율, 학과 단위 부트스트랩 95% 구간, RANDOM_STATE 고정)
# TODO

## ⑤ 시각화 (15점)

In [ ]:
# 프롬프트 ⑤ (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] 분석 표본과 ④ 결과
# [작업] 차트 3종을 그리고 figures/에 저장
# [기준] 축에 단위(명/%) 표기, 절대와 비중을 섞지 않음, 출처 문구 포함
# [보고] 그림 파일명과 한 줄 해석

In [ ]:
# ⑤-1 탈락 인원(명) × 탈락률(%) 산점도 — 소규모 학과 비율 과대 확인
# TODO: 그림 저장 → figures/

In [ ]:
# ⑤-2 계열·설립유형·지역별 고위험 비율 막대 — 지원 대상 기준
# TODO: 그림 저장 → figures/

In [ ]:
# ⑤-3 모델 vs 기준선 적중률·임계값 비교
# TODO: 그림 저장 → figures/

## ⑥ 결론·한계 (15점)

In [ ]:
# 프롬프트 ⑥ (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] H1~H4 판정 결과와 민감도 표
# [작업] 가설별 맞았다/틀렸다, 권고, 근거 수치(셀 번호)를 표로 정리
# [기준] 반대 결과 수용. 한계 3종 필수. 절대=인원, 비중=비율
# [보고] 가설 | 판정 | 근거(셀) | 권고 | 한계

# TODO: 판정 표 출력

### 한계 (빠뜨리면 감점)
1. 테스트는 공시 2026 한 해뿐이다.
2. 학과 단위 집계라 개인 이탈 요인을 담지 못하며, 결과는 상관이지 인과가 아니다.
3. 폐지·통폐합·신설 학과가 연결에서 빠지는 생존 편향과 공시 정의·결측 패턴 변화.
- H4: 한국외대 학과코드 변동(군 단위 풀링), 캠퍼스 구분 불가, 전국 비교군 작음.

## ⑦ 재현성

In [ ]:
# 프롬프트 ⑤ (9주차 4요소 틀 — 원문은 docs/프롬프트_모음.md)
# [대상] 노트북 전체
# [작업] 런타임 재시작 후 처음부터 모두 실행해 핵심 수치를 한 셀에 모아 출력
# [기준] 재현성_확인표의 항목과 같은 이름. 값이 다르면 발견으로 기록
# [보고] 최종 행 수, 분석 표본 행 수, 모델 지표 1~2개, 기준선 값, 실행 일시

# TODO: 핵심 수치 요약 출력 → logs/재현성_확인표.md